In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

Using TensorFlow backend.
/opt/conda/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/opt/conda/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/opt/conda/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/opt/conda/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:519: FutureWarn

In [2]:
train = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [3]:
train.shape,test.shape

((42000, 785), (28000, 784))

In [4]:
train.label.nunique()

10

In [5]:
train = np.array(train,dtype='float32')
test = np.array(test,dtype='float32')
                 
train.shape, test.shape

((42000, 785), (28000, 784))

In [6]:
train_X = train[:,1:] / 255
test_X =  test[:,0:] / 255

train_X = train_X.reshape(train_X.shape[0], 28,28)
test_X = test_X.reshape(test_X.shape[0], 28,28)

train_y = train[:,0]


In [7]:
train_X = train_X.reshape(-1, 28,28, 1)
test_X = test_X.reshape(-1, 28,28, 1)

train_Y_one_hot = to_categorical(train_y)

train_X.shape, test_X.shape,train_Y_one_hot.shape

((42000, 28, 28, 1), (28000, 28, 28, 1), (42000, 10))

In [8]:
X_train,X_valid,y_train,y_valid = train_test_split(train_X,train_Y_one_hot,test_size=0.3)
X_train.shape,X_valid.shape,y_train.shape,y_valid.shape

((29400, 28, 28, 1), (12600, 28, 28, 1), (29400, 10), (12600, 10))

In [9]:
from keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D
from keras.models import Sequential
from keras.layers.normalization import BatchNormalization
from keras.layers.advanced_activations import LeakyReLU
from keras.losses import categorical_crossentropy
from keras.optimizers import Adam

model = Sequential()
model.add(Conv2D(32,kernel_size=(3,3),activation='linear',input_shape=(28,28,1),padding='same'))
model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D((2,2),padding='same'))

model.add(Conv2D(64,kernel_size=(3,3),activation='linear',input_shape=(28,28,1),padding='same'))
model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D((2,2),padding='same'))

model.add(Conv2D(128,kernel_size=(3,3),activation='linear',input_shape=(28,28,1),padding='same'))
model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D((2,2),padding='same'))

model.add(Flatten())

model.add(Dense(128,activation='linear'))
model.add(LeakyReLU(alpha=0.1))
model.add(Dense(10,activation='softmax'))

In [10]:
model.compile(loss='categorical_crossentropy',metrics=['accuracy'],optimizer='Adam')

In [11]:
model_train = model.fit(X_train,y_train)

Epoch 1/1
29400/29400 [==============================] - 28s 949us/step - loss: 0.1770 - acc: 0.9446


In [12]:

test_eval = model.evaluate(X_valid,y_valid,verbose=0)

In [13]:
print("Loss=",test_eval[0])
print("Accuracy=",test_eval[1])

Loss= 0.060256909012557966
Accuracy= 0.981746031783876


In [14]:
batch_size = 64
epochs = 20
num_classes = 10

In [15]:
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),activation='linear',padding='same',input_shape=(28,28,1)))
model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D((2, 2),padding='same'))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), activation='linear',padding='same'))
model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D(pool_size=(2, 2),padding='same'))
model.add(Dropout(0.25))
model.add(Conv2D(128, (3, 3), activation='linear',padding='same'))
model.add(LeakyReLU(alpha=0.1))                  
model.add(MaxPooling2D(pool_size=(2, 2),padding='same'))
model.add(Dropout(0.4))
model.add(Flatten())
model.add(Dense(128, activation='linear'))
model.add(LeakyReLU(alpha=0.1))           
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))

In [16]:
model.compile(loss='categorical_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [17]:
model_dropout = model.fit(X_train,y_train, batch_size=batch_size,epochs=epochs,verbose=1,validation_data=(X_valid, y_valid))

Train on 29400 samples, validate on 12600 samples
Epoch 1/20
29400/29400 [==============================] - 32s 1ms/step - loss: 0.3818 - acc: 0.8764 - val_loss: 0.0891 - val_acc: 0.9745
Epoch 2/20
29400/29400 [==============================] - 32s 1ms/step - loss: 0.1067 - acc: 0.9656 - val_loss: 0.0568 - val_acc: 0.9828
Epoch 3/20
29400/29400 [==============================] - 32s 1ms/step - loss: 0.0776 - acc: 0.9763 - val_loss: 0.0456 - val_acc: 0.9858
Epoch 4/20
29400/29400 [==============================] - 31s 1ms/step - loss: 0.0635 - acc: 0.9807 - val_loss: 0.0563 - val_acc: 0.9845
Epoch 5/20
29400/29400 [==============================] - 32s 1ms/step - loss: 0.0540 - acc: 0.9824 - val_loss: 0.0425 - val_acc: 0.9866
Epoch 6/20
29400/29400 [==============================] - 31s 1ms/step - loss: 0.0506 - acc: 0.9843 - val_loss: 0.0417 - val_acc: 0.9867
Epoch 7/20
29400/29400 [==============================] - 31s 1ms/step - loss: 0.0408 - acc: 0.9873 - val_loss: 0.0372 - val_acc

In [18]:
test_eval = model.evaluate(X_valid, y_valid, verbose=1)

12600/12600 [==============================] - 3s 269us/step


In [19]:
print("Loss=",test_eval[0])
print("Accuracy=",test_eval[1])

Loss= 0.034861775695107
Accuracy= 0.9903968254346697


In [20]:
predicted_classes = model.predict(test_X)
predicted_classes = np.argmax(np.round(predicted_classes),axis=1)
#predicted_classes.shape, test_y.shape

In [21]:
#predicted_classes = network.predict_classes(final_test)

#y_pred = network.predict(final_test)
#predicted_classes = np.argmax(y_pred,axis=1)

submissions=pd.DataFrame({"ImageId": list(range(1,len(predicted_classes)+1)),
                         "Label": predicted_classes})
submissions.to_csv("my_digit.csv", index=False, header=True)